# **0 <font color='orange'>|</font> Install & Import**
---

In [ ]:
# Install
!pip install -q openai --upgrade
!pip install -U langchain-openai

In [ ]:
# Import
import pandas as pd
import numpy as np
import os

from sklearn.metrics.pairwise import cosine_similarity

from google.colab import userdata

from langchain_openai import OpenAIEmbeddings

In [ ]:
# Warnung ausstellen
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Setzen des OpenAI API-Schlüssels
KEY = userdata.get('OpenAI-Kurs-API')
os.environ["OPENAI_API_KEY"] = KEY

# **1 <font color='orange'>|</font> Understand**
---

In [ ]:
# Texten aus Excel-Tabelle einlesen
df = pd.read_excel('/content/Lerninhalte.xlsx')

In [ ]:
df.columns

# **2 <font color='orange'>|</font> Prepare**
---

In [ ]:
# Übernahme einer Textspalte für die späteren Embeddings
# Spalten ['Nr.', 'Kurstitel', 'Lerninhalt (Beschreibung)', 'Text', 'Link', 'Unnamed: 5'],
texts = df['Kurstitel'].tolist()

# **3 <font color='orange'>|</font> Modelling**
---

In [ ]:
# Auswahl des Embeddingsmodells, 512 Dimensionen
emb_model = "text-embedding-3-small"

In [ ]:
# Erstellen einer Embedding-Funktion
embeddings = OpenAIEmbeddings(model=emb_model)

In [ ]:
# Funktion zur Erstellung von Embeddings für eine Liste von Texten
def get_embeddings_for_texts(texts):
    return [embeddings.embed_query(text) for text in texts]

In [ ]:
# Embeddings für die Texte erstellen
%%time
embeddings_list = get_embeddings_for_texts(texts)

In [ ]:
# Embeddings in eine DataFrame speichern und in eine Excel-Datei exportieren
df_emb = pd.DataFrame({'Text': texts, 'Embedding': embeddings_list})
df_emb.to_excel('/content/df_embeddings.xlsx', index=False)

# **4 <font color='orange'>|</font> Evaluate**
---

In [ ]:
# Kategorien einlesen
df_cat = pd.read_excel('/content/Kategorien.xlsx')
categories = df_cat['Kategorie'].tolist()

In [ ]:
# Schwellenwert für die Ähnlichkeit
threshold = 0.4

# DataFrame für das Ergebnis erstellen
result_df = pd.DataFrame()

for category in categories:
    # Schritt 1: Kategorie in ein Embedding umwandeln
    text_input_embedding = embeddings.embed_query(category)

    # Schritt 2: Ähnlichkeit berechnen
    embeddings_matrix = np.array(df_emb['Embedding'].tolist())
    text_input_embedding_matrix = np.array([text_input_embedding])
    similarities = cosine_similarity(text_input_embedding_matrix, embeddings_matrix)

    # Schritt 3: Ähnliche Texte finden und in einem DataFrame speichern
    similarities_df = pd.DataFrame(similarities.T, columns=['Similarity'])

    # Schritt 4: Texte auswählen, deren Ähnlichkeitswert über dem Schwellenwert liegt
    top_x = df_emb[similarities_df['Similarity'] > threshold]

    # Schritt 4: Die entsprechenden Texte und ihre Ähnlichkeiten zusammen mit der Kategorie hinzufügen
    top_x['Category'] = category
    result_df = pd.concat([result_df, top_x], ignore_index=True)

In [ ]:
result_df.shape

In [ ]:
# DataFrame als Excel-Tabelle speichern
result_df.to_excel('/content/result_df.xlsx', columns=['Text', 'Category'], index=False)

# **5 <font color='orange'>|</font> Deploy**
---